In [ ]:
import llc_cutout_dataloader.cutouts_dataset as cutouts_dataset
import visualization.visualization as vis
import dl_embedding.vae as vae
from cuml.cluster import HDBSCAN

import torch
import numpy as np
from matplotlib import pyplot as plt


In [4]:
source = cutouts_dataset.CutoutDataSource(bucket="dbof", folder="cutouts_dataset_v2",
                                          run_id="1_00",
                                          dataset_name="cutout_dataset.zarr",
                                          s3_endpoint="https://s3-west.nrp-nautilus.io"
                                          )

source.print_available_channels()

23 available channels:
['Eta','Salt','Theta','U','V','W','gradb2','oceTAUX','oceTAUY','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','coriolis_f','oceQnet','ekman_pumping','wind_stress_curl','SIarea','XC','YC']


In [5]:
#data_channels = ['Eta','Salt','Theta','U','V','W','gradb2','oceTAUX','oceTAUY','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','coriolis_f','oceQnet','ekman_pumping','wind_stress_curl']
#data_channels_high_res = ['Salt','Theta','U','V','gradb2','oceTAUX','oceTAUY','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','oceQnet','ekman_pumping','wind_stress_curl']
data_channels_no_cor = ['Eta','Salt','Theta','U','V','W','gradb2','oceTAUX','oceTAUY','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','oceQnet','ekman_pumping','wind_stress_curl']

data_channels_engineered = ['gradb2','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','oceQnet','ekman_pumping','wind_stress_curl']
dataset = cutouts_dataset.CutoutDataset.from_source(data_channels=data_channels_engineered, source=source, subset=False,
                                                    subsample_per_chunk=64, num_sample_chunks=1, n_workers=4)

/home/jovyan/conda_envs/main_cuml/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35165 instead
  warnings.warn(


<Client: 'tcp://127.0.0.1:45005' processes=4 threads=32, memory=128.00 GiB>
nrp link url : https://jupyterhub-west.nrp-nautilus.io/hub/user-redirect/proxy/35165/status
dropped 0 ice, 0 NaN; kept 3250 / 3250
features ['gradb2', 'gradrho2', 'turner_angle', 'strain_n', 'strain_s', 'strain_mag', 'divergence', 'relative_vorticity', 'oceQnet', 'ekman_pumping', 'wind_stress_curl'] | coords ['XC', 'YC']


In [ ]:
# cutouts as VAE input (same preprocessing as DINO); patch_size fixed at 8 by the 3x stride-2 encoder
patch_size = 8
images = dataset.preprocess_for_training()   # (N, C, H, W): log-grads + z-score
print(images.shape)


In [ ]:
# VAE embedding step: train on cutouts, embed = per-patch latent means
embedder = vae.ConvVAEEmbedder(latent_dim=16)
embedder.fit(images, epochs=30)
embeddings = embedder.embed(images)          # (N_patches, latent_dim), N_patches = N * (H/8)^2
print(embeddings.shape)


In [ ]:
# Cluster the learned embeddings with cuML HDBSCAN
clusterer = HDBSCAN(min_cluster_size=5, min_samples=None)
clusters = np.asarray(clusterer.fit_predict(embeddings))   # (N_patches,); -1 = noise
print("clusters", int(clusters.max()) + 1,
      "| noise", int((clusters == -1).sum()), "/", clusters.size)


In [ ]:
# 3D view of the clustering (PCA to 3D for display only)
from sklearn.decomposition import PCA
emb3 = PCA(n_components=3).fit_transform(embeddings)
vis.vis_dim_redux(emb3, labels=clusters, dims=3, alpha=0.5)


In [ ]:
np.unique(clusters)


In [ ]:
vis.plot_global_cluster_maps(dataset, clusters, patch_size=patch_size, alpha=0.1,
                             point_size=10.0, extent=None, coastlines=True,
                             panel_size=8, drop_noise=False, save_dir=None)


In [ ]:
vis.make_image_from_patches(dataset, clusters, patch_size=patch_size, number_rows=100)
